# K562 bs128 + 早停训练与插补质量评价

本 notebook 针对当前代码版本（scHiC-Diff 项目内）实现两件事：

1. **训练**：在 K562 HiCImputeData 12 个条件（T1/T2/T3 × 1k/2k/4k/7k）上，以 **batch_size=128 + EarlyStopping(patience=25)** 从零训练扩散模型，并在训练后自动跑 test 推理，输出反归一化插补结果 `denoise_recon_inv.npz`。
2. **插补质量评价**：用 GT / observed / 预测计算逐细胞 **PCC / MAE / SCC**，并按 `all / obs / held` 三个子集汇总，与官方 pipeline 结果核对。

### 为什么需要 CLI 覆盖早停参数

- `main.py` 中 EarlyStopping 的 `patience` 硬编码为 **3000**（≈不早停），训练必须跑满 `max_epochs=1000`；
  本 notebook 用 CLI 覆盖为 `patience=25`，与官方 Batch6/10 协议一致（通常 epoch 90~160 停止）。
- 默认每个 epoch 都写 ~450MB checkpoint；这里用 `lightning.modelcheckpoint.params.every_n_epochs=50` 降低 NFS 写入压力。
- 数据只有 100 cells（80 训练），bs128 与 bs1024 都是 1 step/epoch，结果几乎一致。

### 运行环境

- **提交训练**：登录节点即可（仅调用 `sbatch`），训练在 GPU 节点执行。
- **指标计算**：数据很小（12 × 100 × 1830），可直接在 notebook 内运行；建议 kernel 使用
  `2_schic-scvi-3d`（官方指标环境）或 `scdiff2`。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
from scipy.stats import spearmanr

# ================= 仓库定位（notebook 放在 <repo>/examples/ 下） =================
_here = Path.cwd()
if (_here / "main.py").exists():
    REPO = _here
elif (_here.parent / "main.py").exists():
    REPO = _here.parent
else:
    REPO = next((p for p in _here.parents if (p / "main.py").exists()), _here.parent)
REPO = REPO.resolve()

# ================= 数据路径（项目内相对路径，可用环境变量覆盖） =================
INPUT_DIR = Path(os.environ.get(
    "K562_INPUT_DIR",
    REPO / "5_baseline/7_scHiCDiff/1_HiCImputeData/input"))
GT_DIR = Path(os.environ.get(
    "K562_GT_DIR",
    REPO / "5_baseline/0_gtData/1_Gt_HiCImputeData"))
OBS_DIR = Path(os.environ.get(
    "K562_OBS_DIR",
    REPO / "5_baseline/0_gtData/0_downsampled_HiCImputeData"))
PIPELINE_CSV = REPO / "results/metrics_v5fast_bs128/1_HiCImputedData/HiCImputeData_PCC_MAE_SCC_metrics.csv"

# ================= 运行与输出路径 =================
_default_py = "/public/home/hpc254701055/micromamba/envs/scdiff2/bin/python"
PYTHON_BIN = os.environ.get(
    "PYTHON_BIN",
    _default_py if Path(_default_py).exists() else sys.executable)
SAVE_ROOT = REPO / "results/training_results_v5fast_bs128"   # 插补结果输出目录
LOG_ROOT = REPO / "logs/recon_masked_v5fast_bs128"           # 训练日志目录
EVAL_OUT = REPO / "results/metrics_v5fast_bs128_notebook"    # notebook 指标输出目录

DATASETS = [
    "K562_T1_1k", "K562_T1_2k", "K562_T1_4k", "K562_T1_7k",
    "K562_T2_1k", "K562_T2_2k", "K562_T2_4k", "K562_T2_7k",
    "K562_T3_1k", "K562_T3_2k", "K562_T3_4k", "K562_T3_7k",
]
N_CELLS, N_FEATURES = 100, 1830
ES_PATIENCE = 25      # 与官方 Batch6/10 一致
CKPT_EVERY = 50       # 降低 checkpoint 写入频率

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"{len(DATASETS)} datasets | repo = {REPO}")
print(f"input dir = {INPUT_DIR}")
print(f"gt dir    = {GT_DIR}")
print(f"obs dir   = {OBS_DIR}")
print(f"save root = {SAVE_ROOT}")

## 1. 提交训练（bs128 + 早停）

下面的脚本把所有超参写成 `key=value` 形式的 CLI 覆盖（main.py 会用 OmegaConf dotlist 合并到
`configs/recon_masked.yaml` 上）：

| 覆盖项 | 值 | 作用 |
|---|---|---|
| `data.params.batch_size` | 128 | bs128 训练 |
| `data.params.test_batch_size` | 9999 | 整个测试集一次前向 |
| `lightning.callbacks.early_stopping_callback.params.patience` | 25 | 早停（官方协议） |
| `lightning.modelcheckpoint.params.every_n_epochs` | 50 | checkpoint 降频 |
| `data.params.num_workers` | 4 | 避免节点 CPU 超载 |
| `--save_path` | `results/training_results_v5fast_bs128/{dataset}_sim` | 插补结果目录 |

脚本自带 **skip 逻辑**：若 `{dataset}_sim/denoise_recon_inv.npz` 已存在则跳过，可安全重复提交。
将下方 `SUBMIT` 改为 `True` 即提交 12 任务 array（每任务 1 GPU）。

In [ ]:
SBATCH_TEMPLATE = r"""#!/usr/bin/env bash
#SBATCH -J k562_bs128_nb
#SBATCH -A pi_limin_r
#SBATCH -p gpu2Q,gpu4Q,gpu8Q
#SBATCH --qos=gpuq
#SBATCH --cpus-per-task=10
#SBATCH --gres=gpu:1
#SBATCH --mem=40G
#SBATCH --spread-job
#SBATCH --exclude=gpu207
#SBATCH --array=0-11%12
#SBATCH --output=@LOG_ROOT@/k562_bs128_nb_%A_%a.out
#SBATCH --error=@LOG_ROOT@/k562_bs128_nb_%A_%a.err

set -euo pipefail

REPO="@REPO@"
INPUT_DIR="@INPUT_DIR@"
PYTHON_BIN="@PYTHON_BIN@"
SAVE_ROOT="@SAVE_ROOT@"

DATASETS=(
  K562_T1_1k K562_T1_2k K562_T1_4k K562_T1_7k
  K562_T2_1k K562_T2_2k K562_T2_4k K562_T2_7k
  K562_T3_1k K562_T3_2k K562_T3_4k K562_T3_7k
)

DS="${DATASETS[${SLURM_ARRAY_TASK_ID:?}]}"
H5AD="${INPUT_DIR}/${DS}_sim.h5ad"
SAVE="${SAVE_ROOT}/${DS}_sim"

if [[ -f "${SAVE}/denoise_recon_inv.npz" ]]; then
  echo "[SKIP] already done: ${SAVE}"
  exit 0
fi

mkdir -p "${SAVE}"
cd "${REPO}"

export OMP_NUM_THREADS=4
export OPENBLAS_NUM_THREADS=4
export MKL_NUM_THREADS=4
export NUMEXPR_NUM_THREADS=4

"${PYTHON_BIN}" main.py \
  -t True \
  --base configs/recon_masked.yaml \
  -n "${DS}_sim.seed10" \
  -l logs/recon_masked_v5fast_bs128 \
  --save_path "${SAVE}" \
  data.params.batch_size=128 \
  data.params.test_batch_size=9999 \
  data.params.num_workers=4 \
  lightning.callbacks.early_stopping_callback.params.patience=@ES_PATIENCE@ \
  lightning.modelcheckpoint.params.every_n_epochs=@CKPT_EVERY@ \
  data.params.train.params.dataset=K562 \
  data.params.train.params.fname="${H5AD}" \
  data.params.validation.params.dataset=K562 \
  data.params.validation.params.fname="${H5AD}" \
  data.params.test.params.dataset=K562 \
  data.params.test.params.fname="${H5AD}"

echo "DONE ${DS} $(date)"
"""

SCRIPT_PATH = REPO / "notebook_k562_bs128_es.sbatch"
script = (SBATCH_TEMPLATE
          .replace("@REPO@", str(REPO))
          .replace("@INPUT_DIR@", str(INPUT_DIR))
          .replace("@PYTHON_BIN@", PYTHON_BIN)
          .replace("@SAVE_ROOT@", str(SAVE_ROOT))
          .replace("@LOG_ROOT@", str(LOG_ROOT))
          .replace("@ES_PATIENCE@", str(ES_PATIENCE))
          .replace("@CKPT_EVERY@", str(CKPT_EVERY)))
SCRIPT_PATH.write_text(script)

SUBMIT = False   # <- 改为 True 提交训练
jobid = None
if SUBMIT:
    r = subprocess.run(["sbatch", "--parsable", str(SCRIPT_PATH)],
                       capture_output=True, text=True, check=True)
    jobid = r.stdout.strip()
    print("submitted job:", jobid)
else:
    print("SUBMIT=False，未提交。训练脚本已写入:")
    print(" ", SCRIPT_PATH)
    print("手动提交: sbatch", SCRIPT_PATH)

## 2. 监控训练

- `metrics.csv` 里 `val/loss_MSE_ema` 是 EarlyStopping 监控的指标（`main.py:565` 硬编码 monitor）。
- 早停触发条件：连续 `patience=25` 个 epoch 改善 < `min_delta=1e-4`。
- 每个任务的 test 阶段会用 EMA 权重跑 1000 步扩散采样，结束后在 `{dataset}_sim/` 写出 4 个 npz。

In [ ]:
def latest_log_dir(ds):
    dirs = sorted(LOG_ROOT.glob(f"{ds}_sim.seed10_*"))
    return dirs[-1] if dirs else None


def read_metrics(ds):
    d = latest_log_dir(ds)
    if d is None:
        return None
    f = d / "csv" / "version_0" / "metrics.csv"
    return pd.read_csv(f) if f.exists() else None


def status_table():
    rows = []
    for ds in DATASETS:
        npz = SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz"
        df = read_metrics(ds)
        last_epoch, best = 0, np.nan
        if df is not None:
            val = df[["epoch", "val/loss_MSE_ema"]].dropna()
            if len(val):
                last_epoch = int(val["epoch"].max())
                best = float(val["val/loss_MSE_ema"].min())
        rows.append({
            "dataset": ds,
            "epoch": last_epoch,
            "best_val_MSE_ema": round(best, 5) if best == best else np.nan,
            "imputed": "OK" if npz.exists() else "",
        })
    return pd.DataFrame(rows)


if jobid:
    print(subprocess.run(["squeue", "-j", jobid, "-o", "%.12i %.14j %.8T %.10M %.20R"],
                         capture_output=True, text=True).stdout)

display(status_table())

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, ds in zip(axes.ravel(), DATASETS):
    df = read_metrics(ds)
    if df is None:
        ax.set_title(f"{ds} (no log)", fontsize=9)
        continue
    val = df[["epoch", "val/loss_MSE_ema"]].dropna()
    if len(val):
        ax.plot(val["epoch"], val["val/loss_MSE_ema"], lw=1.2, color="#4c72b0")
        best_ep = int(val.loc[val["val/loss_MSE_ema"].idxmin(), "epoch"])
        ax.axvline(best_ep, color="#c44e52", ls="--", lw=0.8)
        ax.set_title(f"{ds} | best e={best_ep}, stop e={int(val['epoch'].max())}", fontsize=8)
    ax.set_xlabel("epoch", fontsize=8)
    ax.set_ylabel("val/loss_MSE_ema", fontsize=8)
fig.suptitle("Validation loss (MSE EMA), dashed = best epoch", y=1.0)
plt.tight_layout()
plt.show()

## 3. 插补质量评价

**输入**（每个数据集）：

| 数据 | 路径 | 说明 |
|---|---|---|
| GT | `5_baseline/0_gtData/1_Gt_HiCImputeData/{dataset}_true.npz` | 仿真真值 (100, 1830)，三角特征顺序与 observed 一致 |
| observed | `5_baseline/0_gtData/0_downsampled_HiCImputeData/{dataset}_sim.npz` | 观测输入（有缺失，亦支持从子目录原始 txt 直接读取） |
| 预测 | `results/training_results_v5fast_bs128/{dataset}_sim/denoise_recon_inv.npz` | 反归一化插补结果（raw scale） |

**指标定义**（与官方 `paperplots/1_pccAndMae_all/calculate_imputation_metrics.py` 完全一致）：

- 逐细胞计算 PCC / MAE / SCC，再跨细胞取 `nanmean` / `nanstd`；
- 三个子集：
  - `all`：全部 1830 个三角特征（含 GT=0 的位置）；
  - `obs`：observed > 0；
  - `held`：GT > 0 且 observed ≤ 0（真正"插补"出来的位置）；
- 直接在原始数值上计算，**不做 log/clip/归一化**（负预测值保留）。

In [ ]:
def load_matrix(path):
    path = Path(path)
    if path.is_file():
        m = load_npz(path)
        if hasattr(m, "toarray"):
            m = m.toarray()
        return np.asarray(m, dtype=np.float64)
    if path.is_dir():
        tril_r, tril_c = np.tril_indices(61, k=-1)
        cells = []
        for i in range(1, 101):
            f = path / f"cell_{i}_chr19.txt"
            mat = np.zeros((61, 61), dtype=np.float64)
            if f.exists() and f.stat().st_size > 0:
                data = np.loadtxt(f, dtype=int)
                if data.ndim == 1 and data.size == 3:
                    mat[data[0], data[1]] = data[2]
                elif data.ndim == 2:
                    mat[data[:, 0], data[:, 1]] = data[:, 2]
            cells.append(mat[tril_r, tril_c])
        return np.asarray(cells, dtype=np.float64)
    stem = path.stem.removesuffix("_sim")
    alt_dir = path.parent / stem
    if alt_dir.is_dir():
        return load_matrix(alt_dir)
    raise FileNotFoundError(f"未找到数据文件或目录: {path}")


def safe_pearson(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size == 0 or b.size == 0 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def safe_mae(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size == 0 or b.size == 0:
        return np.nan
    return float(np.mean(np.abs(a - b)))


def safe_spearman(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(spearmanr(a, b)[0])


def evaluate_one(ds):
    gt = load_matrix(GT_DIR / f"{ds}_true.npz")
    obs = load_matrix(OBS_DIR / f"{ds}_sim.npz")
    pred = load_matrix(SAVE_ROOT / f"{ds}_sim" / "denoise_recon_inv.npz")
    assert gt.shape == obs.shape == pred.shape == (N_CELLS, N_FEATURES), \
        f"shape mismatch: gt={gt.shape}, obs={obs.shape}, pred={pred.shape}"

    per_cell = {f"{m}_{s}": [] for m in ("pcc", "mae", "scc") for s in ("all", "obs", "held")}
    n_held = []
    for i in range(gt.shape[0]):
        g, o, p = gt[i], obs[i], pred[i]
        masks = {
            "all": np.ones_like(g, dtype=bool),
            "obs": o > 0,
            "held": (g > 0) & ~(o > 0),
        }
        n_held.append(int(masks["held"].sum()))
        for s, m in masks.items():
            per_cell[f"pcc_{s}"].append(safe_pearson(p[m], g[m]))
            per_cell[f"mae_{s}"].append(safe_mae(p[m], g[m]))
            per_cell[f"scc_{s}"].append(safe_spearman(p[m], g[m]))

    row = {"data_name": ds, "ctype": ds.split("_")[1], "cdepth": ds.split("_")[2]}
    for k, v in per_cell.items():
        row[f"{k}_mean"] = float(np.nanmean(v))
        row[f"{k}_std"] = float(np.nanstd(v))
    row["n_held_mean"] = float(np.mean(n_held))
    return row, per_cell


In [ ]:
if PIPELINE_CSV.exists():
    pipe = pd.read_csv(PIPELINE_CSV)
    pipe = pipe[pipe["method"] == "scHiC-Diff"][
        ["data_name", "pcc_all_mean", "pcc_held_mean", "mae_all_mean"]]
    mine = metrics_df[["data_name", "pcc_all_mean", "pcc_held_mean", "mae_all_mean"]]
    cmp = pipe.merge(mine, on="data_name", suffixes=("_pipeline", "_notebook"))
    display(cmp.round(4))
    for col in ("pcc_all_mean", "pcc_held_mean", "mae_all_mean"):
        diff = (cmp[f"{col}_pipeline"] - cmp[f"{col}_notebook"]).abs().max()
        print(f"max |pipeline - notebook| {col}: {diff:.2e}")
else:
    print("未找到 pipeline 指标 CSV（可先运行 results/metrics_v5fast_bs128 下的 prepare/array/aggregate）:",
          PIPELINE_CSV)

In [ ]:
plot_df = metrics_df.set_index("data_name")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, key, title in zip(
    axes,
    ["pcc_all_mean", "pcc_held_mean", "mae_all_mean"],
    ["PCC (all features)", "PCC (held-out)", "MAE (all features)"],
):
    err = plot_df[key.replace("_mean", "_std")]
    ax.bar(plot_df.index, plot_df[key], yerr=err, capsize=2, color="#4c72b0", alpha=0.9)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=90, labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
DS_DETAIL = "K562_T1_2k"
pc = per_cell_store[DS_DETAIL]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for s, c in [("all", "#4c72b0"), ("obs", "#55a868"), ("held", "#c44e52")]:
    v = np.asarray(pc[f"pcc_{s}"], dtype=float)
    v = np.sort(v[~np.isnan(v)])
    axes[0].plot(np.arange(len(v)), v, marker=".", ms=3, lw=0.8, color=c, label=s)
axes[0].set_xlabel("cell (sorted)")
axes[0].set_ylabel("PCC")
axes[0].set_title(f"Per-cell PCC by subset - {DS_DETAIL}")
axes[0].legend()
axes[0].set_ylim(0, 1.02)

gt = load_matrix(GT_DIR / f"{DS_DETAIL}_true.npz")[0]
obs = load_matrix(OBS_DIR / f"{DS_DETAIL}_sim.npz")[0]
pred = load_matrix(SAVE_ROOT / f"{DS_DETAIL}_sim" / "denoise_recon_inv.npz")[0]
held = (gt > 0) & ~(obs > 0)
axes[1].scatter(gt[~held], pred[~held], s=6, alpha=0.3, label="other features")
axes[1].scatter(gt[held], pred[held], s=8, alpha=0.7, color="#c44e52",
                label="held-out (GT>0, obs=0)")
lim = max(gt.max(), pred.max()) * 1.02
axes[1].plot([0, lim], [0, lim], "k--", lw=0.8)
axes[1].set_xlabel("GT (raw)")
axes[1].set_ylabel("prediction (raw)")
axes[1].set_title(f"Cell 0: prediction vs GT - {DS_DETAIL}")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. 说明与复现

**输出位置**

```text
results/training_results_v5fast_bs128/{dataset}_sim/
├── raw_x.npz              # 观测输入（有缺失，无归一化）
├── denoise_recon.npz      # 归一化空间预测
├── denoise_recon_inv.npz  # ★ 反归一化插补结果（评价用）
└── denoise_target.npz     # 测试 target（归一化）
logs/recon_masked_v5fast_bs128/{dataset}_sim.seed10_<timestamp>/
results/metrics_v5fast_bs128_notebook/HiCImputeData_PCC_MAE_SCC_metrics.csv
```

**复现 bs1024 对照**

只需把提交脚本里的 `data.params.batch_size=128` 改为 `1024`、`data.params.test_batch_size=9999` 改为 `1024`，
并把 `SAVE_ROOT` / 日志目录换名（参考仓库内 `submit_k562_bs1024_es.sbatch`）。
注意：本数据集只有 80 个训练 cells，bs128 与 bs1024 都是 1 step/epoch，两者结果应几乎一致。

**注意事项**

- 训练必须提交 GPU 作业；notebook 内只做提交与监控。
- `gpu207` 节点出现过偶发 SIGKILL，脚本已加 `--exclude=gpu207`；若其他节点出现相同问题，补跑对应 task 即可
  （脚本有 skip 逻辑，直接用 `sbatch --array=<idx>` 重跑）。
- 官方 SLURM 指标 pipeline（133 任务）仍以 `paperplots/1_pccAndMae_all/` 为准；本 notebook 的计算是
  等价的轻量实现，并会与其结果核对（见第 3 节对比单元）。